# Experiment: Recent Top Ladder Deck Archetype Audit

Objective: analyze the latest mounted official top-episode datasets and identify current high-volume teams, exact deck hashes, archetype share, and fast-moving control/LO tech.

This notebook is designed to run on Kaggle. It uses mounted datasets only, writes compact CSV/JSON artifacts to `/kaggle/working`, and does not download replay dumps locally.

Success criterion: export a complete latest-day Top-50 exact-deck table, a two-day archetype trend, effort-aware robustness metrics, visual strength rankings, representative card images, and an archetype matchup matrix.

## Setup

The source window for this run is fixed to the two latest available daily episode datasets: 2026-07-20 and 2026-07-21. Update `TARGET_DATES` when a new analysis window is intended.

In [ ]:
from __future__ import annotations

import ast
import csv
import hashlib
import json
import os
import re
import time
from concurrent.futures import ProcessPoolExecutor
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any

import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 160)

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")
RUNNING_ON_KAGGLE = KAGGLE_INPUT.exists() and KAGGLE_WORKING.exists()
INPUT = KAGGLE_INPUT if RUNNING_ON_KAGGLE else Path.cwd()
WORK = KAGGLE_WORKING if RUNNING_ON_KAGGLE else Path.cwd() / "notebook_outputs"
OUT_DIR = WORK / "recent_top_ladder_deck_archetype_audit"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_DATES = ["2026-07-20", "2026-07-21"]
MAX_REPLAYS_PER_DAY = int(os.environ.get("MAX_REPLAYS_PER_DAY", "0") or "0")
JSON_PARSE_WORKERS = int(os.environ.get("JSON_PARSE_WORKERS", "4") or "4")
MAX_HEADER_BYTES = int(os.environ.get("MAX_HEADER_BYTES", str(256 * 1024)) or str(256 * 1024))
TOP_N_DECK_EXPORT = int(os.environ.get("TOP_N_DECK_EXPORT", "50") or "50")
MIN_DECK_GAMES_FOR_VIEW = int(os.environ.get("MIN_DECK_GAMES_FOR_VIEW", "10") or "10")

print("running_on_kaggle:", RUNNING_ON_KAGGLE)
print("input:", INPUT)
print("out_dir:", OUT_DIR)
print("target_dates:", TARGET_DATES)
print("max_replays_per_day:", MAX_REPLAYS_PER_DAY or "all")
print("json_parse_workers:", JSON_PARSE_WORKERS)
print("max_header_bytes:", MAX_HEADER_BYTES)

## Archetype Classifier

This is a lightweight signature classifier for meta triage. Exact deck hashes remain the source of truth; archetype labels are used to summarize pressure pools and trends.

In [ ]:
ARCHETYPE_SIGNATURES: dict[str, dict[int, float]] = {
    "Marnie's Grimmsnarl ex": {
        646: 1.2,
        647: 1.2,
        648: 2.5,
        112: 0.8,
        1079: 0.6,
        1219: 0.7,
        1259: 0.8,
        7: 0.2,
    },
    "Cubchoo Hammer Control": {
        506: 2.2,
        1120: 1.1,
        1081: 1.1,
        1219: 0.9,
        1166: 0.7,
        1247: 0.7,
        414: 0.5,
        3: 0.2,
    },
    "Mega Kangaskhan Box": {
        756: 2.4,
        1071: 1.3,
        272: 1.0,
        1250: 0.8,
        1198: 0.7,
        1146: 0.5,
        184: 0.5,
    },
    "Chandelure/Comfey": {
        97: 1.2,
        98: 2.0,
        494: 1.0,
        164: 1.0,
        1197: 0.7,
        1225: 0.6,
        1264: 0.5,
        19: 0.3,
    },
    "Okidogi/Solrock": {
        116: 2.0,
        676: 1.1,
        675: 0.8,
        1051: 0.8,
        1052: 1.0,
        1187: 0.7,
        1142: 0.7,
        16: 0.4,
    },
    "Team Rocket Spidops": {
        400: 1.3,
        401: 2.0,
        1134: 1.0,
        1216: 0.8,
        1220: 0.8,
        1218: 0.6,
        1257: 0.6,
        15: 0.4,
    },
    "Cynthia Garchomp": {
        379: 1.2,
        380: 1.2,
        381: 2.2,
        341: 0.8,
        342: 0.8,
        1173: 0.8,
        1141: 0.6,
        20: 0.4,
    },
    "Mega Venusaur ex": {
        650: 1.0,
        651: 1.0,
        652: 2.2,
        96: 0.8,
        1094: 0.7,
        1261: 0.7,
        1: 0.3,
    },
    "Mega Abomasnow ex": {
        721: 0.9,
        722: 1.4,
        723: 2.2,
        1145: 0.8,
        1205: 0.7,
        1227: 0.5,
        1235: 0.8,
        1158: 0.6,
        3: 0.2,
    },
    "Mega Starmie ex": {
        1030: 1.4,
        1031: 2.2,
        666: 0.9,
        1145: 0.8,
        1189: 0.8,
        1229: 0.7,
        1120: 0.5,
        3: 0.4,
        17: 0.4,
    },
    "Mega Lucario ex": {
        673: 1.0,
        674: 1.0,
        675: 0.8,
        676: 0.8,
        677: 1.2,
        678: 2.2,
        1141: 0.7,
        1142: 0.7,
        6: 0.4,
    },
    "Crustle": {
        344: 1.4,
        345: 2.0,
        532: 1.0,
        533: 1.5,
        1257: 0.8,
        1219: 0.7,
        1197: 0.6,
        1186: 0.6,
        1: 0.3,
        14: 0.3,
    },
    "Great Tusk LO": {
        58: 2.2,
        344: 0.8,
        345: 1.1,
        1086: 0.8,
        1182: 0.6,
        1185: 1.3,
        1197: 0.8,
        1247: 1.2,
    },
    "Hop's Trevenant": {
        878: 1.4,
        879: 2.0,
        304: 1.3,
        1115: 0.8,
        1171: 0.8,
        1255: 0.9,
        12: 0.4,
    },
    "Dudunsparce/Alakazam": {
        65: 0.8,
        66: 1.2,
        305: 1.0,
        741: 1.1,
        742: 1.1,
        743: 1.6,
        1079: 0.7,
        1225: 0.5,
        1231: 0.6,
        1264: 0.5,
        5: 0.3,
        19: 0.3,
    },
    "Archaludon/Cinderace": {
        169: 1.5,
        190: 2.0,
        666: 1.0,
        1121: 0.4,
        1122: 0.4,
        1159: 0.4,
        1185: 0.6,
        1244: 0.8,
        8: 0.3,
    },
    "Dragapult ex": {
        119: 0.9,
        120: 1.1,
        121: 2.0,
        1086: 0.4,
        1079: 0.6,
    },
    "Iono's Bellibolt ex": {
        265: 0.8,
        268: 0.8,
        269: 2.0,
        270: 0.8,
        4: 0.4,
    },
}

REQUIRED_CORES: dict[str, set[int]] = {
    "Marnie's Grimmsnarl ex": {648},
    "Cubchoo Hammer Control": {506},
    "Mega Kangaskhan Box": {756},
    "Chandelure/Comfey": {98},
    "Okidogi/Solrock": {116},
    "Team Rocket Spidops": {401},
    "Cynthia Garchomp": {381},
    "Mega Venusaur ex": {652},
    "Mega Abomasnow ex": {722, 723},
    "Mega Starmie ex": {1030, 1031},
    "Mega Lucario ex": {677, 678},
    "Crustle": {345, 533},
    "Great Tusk LO": {58},
    "Hop's Trevenant": {878, 879},
    "Dudunsparce/Alakazam": {743},
    "Archaludon/Cinderace": {169, 190},
    "Dragapult ex": {121},
    "Iono's Bellibolt ex": {269},
}

## Utilities

Replay extraction reads the first visualization action, where both submitted 60-card decks are exposed. The script also works if Kaggle changes folder names, as long as the date appears in the path.

In [ ]:
def parse_int_list(value: Any) -> list[int]:
    if value is None:
        return []
    if isinstance(value, list):
        out: list[int] = []
        for item in value:
            try:
                out.append(int(item))
            except (TypeError, ValueError):
                pass
        return out
    text = str(value).strip()
    if not text:
        return []
    try:
        parsed = json.loads(text)
        if isinstance(parsed, list):
            return parse_int_list(parsed)
    except json.JSONDecodeError:
        pass
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, list):
            return parse_int_list(parsed)
    except (SyntaxError, ValueError):
        pass
    out: list[int] = []
    for part in text.replace("[", "").replace("]", "").split(","):
        part = part.strip().strip('"').strip("'")
        if not part:
            continue
        try:
            out.append(int(part))
        except ValueError:
            continue
    return out


def deck_hash(deck: list[int]) -> str:
    payload = ",".join(str(card) for card in sorted(deck))
    return hashlib.sha1(payload.encode("utf-8")).hexdigest()[:12]


def classify_deck(deck: list[int]) -> tuple[str, float, str]:
    counts = Counter(deck)
    scores: dict[str, float] = {}
    hits: dict[str, list[str]] = {}
    for archetype, signature in ARCHETYPE_SIGNATURES.items():
        required = REQUIRED_CORES.get(archetype)
        if required and not any(counts.get(card_id, 0) > 0 for card_id in required):
            scores[archetype] = 0.0
            hits[archetype] = []
            continue
        score = 0.0
        hit_parts: list[str] = []
        for card_id, weight in signature.items():
            n = counts.get(card_id, 0)
            if n <= 0:
                continue
            score += min(n, 4) * weight
            hit_parts.append(f"{card_id}x{n}")
        scores[archetype] = score
        hits[archetype] = hit_parts
    best = max(scores, key=scores.get)
    if scores[best] <= 0:
        return "Unknown", 0.0, ""
    ordered = sorted(scores.items(), key=lambda item: item[1], reverse=True)
    runner_gap = ordered[0][1] - (ordered[1][1] if len(ordered) > 1 else 0.0)
    label = best if runner_gap >= 0.75 or ordered[0][1] >= 5.0 else f"{best}?"
    return label, round(scores[best], 3), ";".join(hits[best])


def infer_date(path: Path, fallback: str = "") -> str:
    match = re.search(r"20\d{2}-\d{2}-\d{2}", str(path))
    return match.group(0) if match else fallback


def find_manifest() -> Path | None:
    candidates: list[Path] = []
    for root in [INPUT, Path.cwd()]:
        if root.exists():
            candidates.extend(root.rglob("manifest.csv"))
    candidates.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return candidates[0] if candidates else None


def find_card_csv() -> Path | None:
    names = ["EN_Card_Data.csv", "card_data.csv", "cards.csv"]
    candidates: list[Path] = []
    for root in [INPUT, Path.cwd()]:
        if not root.exists():
            continue
        for name in names:
            candidates.extend(root.rglob(name))
    candidates.sort(key=lambda p: (p.name != "EN_Card_Data.csv", len(str(p))))
    return candidates[0] if candidates else None


def load_card_names(path: Path | None) -> dict[int, str]:
    if path is None or not path.exists():
        return {}
    names: dict[int, str] = {}
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            raw_id = row.get("Card ID") or row.get("card_id") or row.get("id") or ""
            try:
                card_id = int(raw_id)
            except ValueError:
                continue
            name = row.get("Card Name") or row.get("card_name") or row.get("name") or f"card_id:{card_id}"
            names.setdefault(card_id, name)
    return names


def selected_daily_roots() -> dict[str, list[Path]]:
    roots_by_date: dict[str, list[Path]] = {date: [] for date in TARGET_DATES}
    base_candidates = [
        INPUT,
        INPUT / "datasets",
        INPUT / "datasets" / "organizations" / "kaggle",
    ]
    for date in TARGET_DATES:
        slug = f"pokemon-tcg-ai-battle-episodes-{date}"
        exact = [base / slug for base in base_candidates if (base / slug).exists()]
        if exact:
            roots_by_date[date].extend(exact)
            continue
        matches: list[Path] = []
        for base in base_candidates:
            if not base.exists():
                continue
            matches.extend(p for p in base.rglob("*") if p.is_dir() and (slug in p.name or date in p.name))
        roots_by_date[date] = sorted(set(matches), key=lambda p: (len(str(p)), str(p)))
    return {date: roots for date, roots in roots_by_date.items() if roots}


def records_from_replay_json_full(path: Path, fallback_date: str = "") -> list[dict[str, Any]]:
    try:
        payload = json.loads(path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return []
    steps = payload.get("steps") or []
    if not steps:
        return []
    first = steps[0][0] if isinstance(steps[0], list) and steps[0] else steps[0]
    visual = (first or {}).get("visualize") or []
    action = visual[0].get("action") if visual and isinstance(visual[0], dict) else None
    if not isinstance(action, list) or len(action) < 2:
        return []
    info = payload.get("info") or {}
    team_names = info.get("TeamNames") or []
    rewards = payload.get("rewards") or []
    episode_id = info.get("EpisodeId") or payload.get("id") or path.stem
    date = infer_date(path, fallback_date)
    records: list[dict[str, Any]] = []
    for player_index in (0, 1):
        deck = parse_int_list(action[player_index])
        if len(deck) != 60:
            continue
        is_win: bool | None = None
        if player_index < len(rewards) and rewards[player_index] is not None:
            try:
                is_win = float(rewards[player_index]) > 0
            except (TypeError, ValueError):
                is_win = None
        archetype, score, hits = classify_deck(deck)
        records.append(
            {
                "episode_id": str(episode_id),
                "date": date,
                "source_path": str(path.relative_to(INPUT)) if str(path).startswith(str(INPUT)) else str(path),
                "player_index": player_index,
                "team": str(team_names[player_index]) if player_index < len(team_names) else "",
                "opponent": str(team_names[1 - player_index]) if len(team_names) > 1 else "",
                "is_win": is_win,
                "deck_hash": deck_hash(deck),
                "archetype": archetype,
                "archetype_score": score,
                "signature_hits": hits,
                "deck": deck,
            }
        )
    return records


def json_array_from_text(text: str, start: int) -> Any | None:
    if start < 0 or start >= len(text) or text[start] != "[":
        return None
    depth = 0
    in_string = False
    escape = False
    for idx in range(start, len(text)):
        ch = text[idx]
        if in_string:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_string = False
            continue
        if ch == '"':
            in_string = True
        elif ch == "[":
            depth += 1
        elif ch == "]":
            depth -= 1
            if depth == 0:
                return json.loads(text[start : idx + 1])
    return None


def json_array_after_pattern(text: str, pattern: str) -> Any | None:
    match = re.search(pattern, text)
    if not match:
        return None
    idx = match.end()
    while idx < len(text) and text[idx].isspace():
        idx += 1
    return json_array_from_text(text, idx)


def read_replay_header(path: Path, max_bytes: int = MAX_HEADER_BYTES) -> str:
    with path.open("rb") as f:
        data = f.read(max_bytes)
    return data.decode("utf-8", errors="ignore")


def make_records_from_decks(
    *,
    path: Path,
    fallback_date: str,
    decks: list[list[int]],
    team_names: list[Any],
    rewards: list[Any],
    episode_id: Any,
) -> list[dict[str, Any]]:
    date = infer_date(path, fallback_date)
    records: list[dict[str, Any]] = []
    for player_index in (0, 1):
        deck = parse_int_list(decks[player_index] if player_index < len(decks) else [])
        if len(deck) != 60:
            continue
        is_win: bool | None = None
        if player_index < len(rewards) and rewards[player_index] is not None:
            try:
                is_win = float(rewards[player_index]) > 0
            except (TypeError, ValueError):
                is_win = None
        archetype, score, hits = classify_deck(deck)
        records.append(
            {
                "episode_id": str(episode_id or path.stem),
                "date": date,
                "source_path": str(path.relative_to(INPUT)) if str(path).startswith(str(INPUT)) else str(path),
                "player_index": player_index,
                "team": str(team_names[player_index]) if player_index < len(team_names) else "",
                "opponent": str(team_names[1 - player_index]) if len(team_names) > 1 else "",
                "is_win": is_win,
                "deck_hash": deck_hash(deck),
                "archetype": archetype,
                "archetype_score": score,
                "signature_hits": hits,
                "deck": deck,
            }
        )
    return records


def records_from_replay_json(path: Path, fallback_date: str = "") -> list[dict[str, Any]]:
    try:
        header = read_replay_header(path)
        action = json_array_after_pattern(header, r'"visualize"\s*:\s*\[\s*\{\s*"action"\s*:')
        team_names = json_array_after_pattern(header, r'"TeamNames"\s*:')
        rewards = json_array_after_pattern(header, r'"rewards"\s*:')
        episode_match = re.search(r'"EpisodeId"\s*:\s*([0-9]+)', header)
        episode_id = episode_match.group(1) if episode_match else path.stem
        if isinstance(action, list) and len(action) >= 2:
            records = make_records_from_decks(
                path=path,
                fallback_date=fallback_date,
                decks=action,
                team_names=team_names if isinstance(team_names, list) else [],
                rewards=rewards if isinstance(rewards, list) else [],
                episode_id=episode_id,
            )
            if len(records) == 2:
                return records
    except Exception:
        pass
    return records_from_replay_json_full(path, fallback_date=fallback_date)


def parse_file_task(item: tuple[str, str]) -> list[dict[str, Any]]:
    path_text, fallback_date = item
    return records_from_replay_json(Path(path_text), fallback_date=fallback_date)


def top_cards_text(deck: list[int], names: dict[int, str], n: int = 12) -> str:
    return "; ".join(
        f"{card}:{names.get(card, str(card))}x{count}"
        for card, count in Counter(deck).most_common(n)
    )


def safe_slug(text: str, max_len: int = 48) -> str:
    slug = re.sub(r"[^A-Za-z0-9]+", "_", text).strip("_")
    return (slug or "deck")[:max_len]

## Discover Mounted Inputs

If the table below does not show the expected latest dates, update notebook metadata to mount the newer daily datasets and rerun.

In [ ]:
mounted_roots = sorted([p for p in INPUT.iterdir() if p.is_dir()]) if INPUT.exists() else []
print("mounted roots:")
for root in mounted_roots:
    print(" -", root)

manifest_path = find_manifest()
print("manifest:", manifest_path)
if manifest_path:
    manifest_df = pd.read_csv(manifest_path)
    display(manifest_df.tail(10))

daily_roots = selected_daily_roots()
print("selected daily roots:")
for date, roots in daily_roots.items():
    json_count = sum(1 for root in roots for _ in root.rglob("*.json"))
    print(date, "json_count=", json_count, "roots=", [str(r) for r in roots])

analysis_dates = [date for date in TARGET_DATES if date in daily_roots]
missing_dates = [date for date in TARGET_DATES if date not in daily_roots]
if missing_dates:
    print("WARNING: missing mounted dates:", missing_dates)
if not analysis_dates:
    raise RuntimeError("No target-date datasets are mounted.")
print("analysis_dates:", analysis_dates)

card_csv = find_card_csv()
card_names = load_card_names(card_csv)
print("card_csv:", card_csv)
print("card_names:", len(card_names))

## Extract Player Deck Records

Each replay contributes two player-deck records. The dataframe is deduplicated by `episode_id`, `player_index`, and `deck_hash`.

In [ ]:
started = time.time()
records: list[dict[str, Any]] = []
day_file_counts: dict[str, int] = {}

for date in analysis_dates:
    roots = daily_roots[date]
    files: list[Path] = []
    for root in roots:
        files.extend(p for p in root.rglob("*.json") if "kernel-metadata" not in p.name)
    files = sorted(files)
    if MAX_REPLAYS_PER_DAY:
        files = files[:MAX_REPLAYS_PER_DAY]
    day_file_counts[date] = len(files)
    tasks = [(str(path), date) for path in files]
    if JSON_PARSE_WORKERS > 1 and len(tasks) > 1:
        with ProcessPoolExecutor(max_workers=JSON_PARSE_WORKERS) as pool:
            for idx, parsed in enumerate(pool.map(parse_file_task, tasks, chunksize=16), start=1):
                records.extend(parsed)
                if idx % 1000 == 0:
                    print(f"{date}: processed {idx}/{len(files)} files, records={len(records)}")
    else:
        for idx, path in enumerate(files, start=1):
            records.extend(records_from_replay_json(path, fallback_date=date))
            if idx % 1000 == 0:
                print(f"{date}: processed {idx}/{len(files)} files, records={len(records)}")

records_df = pd.DataFrame(records)
if records_df.empty:
    raise RuntimeError("No deck records extracted. Check dataset mounts and replay JSON format.")

records_df = records_df.drop_duplicates(["episode_id", "player_index", "deck_hash"]).reset_index(drop=True)
records_df["is_win_num"] = records_df["is_win"].map(lambda x: 1 if x is True else 0)
records_df["known_result"] = records_df["is_win"].map(lambda x: x is True or x is False)
records_df["deck_csv"] = records_df["deck"].map(lambda deck: ",".join(str(card) for card in deck))

records_path = OUT_DIR / "player_deck_records.csv"
records_df.drop(columns=["deck"]).to_csv(records_path, index=False)

print("elapsed_sec:", round(time.time() - started, 2))
print("player_deck_records:", len(records_df))
print("unique_episodes:", records_df["episode_id"].nunique())
print("unique_decks:", records_df["deck_hash"].nunique())
print("records_csv:", records_path)
display(records_df[["date", "team", "opponent", "is_win", "deck_hash", "archetype", "signature_hits"]].head(10))

## Archetype Share and Trend

Use the combined table for current pressure-pool weights and the trend table to catch rapid public meta drift.

In [ ]:
day_totals = records_df.groupby("date").size().rename("day_total").reset_index()

archetype_day = (
    records_df.groupby(["date", "archetype"], dropna=False)
    .agg(
        games=("deck_hash", "size"),
        wins=("is_win_num", "sum"),
        unique_decks=("deck_hash", "nunique"),
        unique_teams=("team", "nunique"),
    )
    .reset_index()
    .merge(day_totals, on="date", how="left")
)
archetype_day["share"] = archetype_day["games"] / archetype_day["day_total"]
archetype_day["win_rate"] = archetype_day["wins"] / archetype_day["games"]
archetype_day = archetype_day.sort_values(["date", "games"], ascending=[False, False])

archetype_combined = (
    records_df.groupby("archetype", dropna=False)
    .agg(
        games=("deck_hash", "size"),
        wins=("is_win_num", "sum"),
        unique_decks=("deck_hash", "nunique"),
        unique_teams=("team", "nunique"),
    )
    .reset_index()
)
archetype_combined["share"] = archetype_combined["games"] / len(records_df)
archetype_combined["win_rate"] = archetype_combined["wins"] / archetype_combined["games"]
archetype_combined = archetype_combined.sort_values("games", ascending=False)

share_pivot = archetype_day.pivot_table(index="archetype", columns="date", values="share", fill_value=0.0)
game_pivot = archetype_day.pivot_table(index="archetype", columns="date", values="games", fill_value=0)
trend = share_pivot.reset_index()
if analysis_dates[0] in trend.columns and analysis_dates[-1] in trend.columns:
    trend["delta_latest_vs_earliest"] = trend[analysis_dates[-1]] - trend[analysis_dates[0]]
else:
    cols = [col for col in analysis_dates if col in trend.columns]
    trend["delta_latest_vs_earliest"] = trend[cols[-1]] - trend[cols[0]] if len(cols) >= 2 else 0.0
trend["combined_games"] = trend["archetype"].map(archetype_combined.set_index("archetype")["games"])
trend = trend.sort_values(["combined_games", "delta_latest_vs_earliest"], ascending=[False, False])

archetype_day.to_csv(OUT_DIR / "archetype_by_day.csv", index=False)
archetype_combined.to_csv(OUT_DIR / "archetype_combined.csv", index=False)
trend.to_csv(OUT_DIR / "archetype_trend.csv", index=False)

display(archetype_combined.head(20))
display(trend.head(25))

## Cross-day Robustness View

This table looks for broad archetypes whose observed result does not depend on one unusually favorable day. It is descriptive evidence from top-episode sampling, not a causal deck ranking or a substitute for local head-to-head evaluation.

In [ ]:
daily_robustness = (
    archetype_day.groupby("archetype", dropna=False)
    .agg(
        days_present=("date", "nunique"),
        min_daily_games=("games", "min"),
        min_daily_win_rate=("win_rate", "min"),
        max_daily_win_rate=("win_rate", "max"),
        mean_daily_win_rate=("win_rate", "mean"),
        daily_wr_std=("win_rate", "std"),
        min_daily_share=("share", "min"),
        max_daily_share=("share", "max"),
        daily_share_std=("share", "std"),
    )
    .reset_index()
)
daily_robustness[["daily_wr_std", "daily_share_std"]] = daily_robustness[["daily_wr_std", "daily_share_std"]].fillna(0.0)

robustness = daily_robustness.merge(
    archetype_combined[["archetype", "games", "share", "win_rate", "unique_decks", "unique_teams"]],
    on="archetype",
    how="left",
)
latest_share_map = archetype_day[archetype_day["date"] == analysis_dates[-1]].set_index("archetype")["share"]
robustness["latest_share"] = robustness["archetype"].map(latest_share_map).fillna(0.0)
robustness["full_window"] = robustness["days_present"] == len(analysis_dates)
robustness["downside_gap"] = robustness["win_rate"] - robustness["min_daily_win_rate"]
robustness = robustness.sort_values(["full_window", "min_daily_win_rate", "games"], ascending=[False, False, False])
robust_candidate_view = robustness[(robustness["full_window"]) & (robustness["share"] >= 0.01)].copy()
robust_candidate_view = robust_candidate_view.sort_values(["min_daily_win_rate", "win_rate", "games"], ascending=False)

robustness.to_csv(OUT_DIR / "archetype_robustness.csv", index=False)
display(robust_candidate_view.head(25))

## Exact Deck Hashes

Exact hashes are the most actionable output. These rows identify lists worth importing into local H2H pressure pools.

In [ ]:
deck_lookup: dict[str, list[int]] = {}
for row in records_df.itertuples():
    deck_lookup.setdefault(row.deck_hash, list(row.deck))

deck_rows: list[dict[str, Any]] = []
for deck_hash_value, group in records_df.groupby("deck_hash"):
    deck = deck_lookup[str(deck_hash_value)]
    archetype_counts = Counter(group["archetype"])
    team_counts = Counter(t for t in group["team"] if t)
    day_counts = Counter(group["date"])
    games = len(group)
    wins = int(group["is_win_num"].sum())
    deck_rows.append(
        {
            "deck_hash": deck_hash_value,
            "archetype": archetype_counts.most_common(1)[0][0],
            "games": games,
            "wins": wins,
            "win_rate": wins / games if games else 0.0,
            "unique_teams": len(team_counts),
            "example_teams": "; ".join(t for t, _ in team_counts.most_common(8)),
            "first_seen": min(day_counts),
            "last_seen": max(day_counts),
            "day_counts": json.dumps(dict(sorted(day_counts.items())), ensure_ascii=False),
            "top_cards": top_cards_text(deck, card_names, n=14),
            "deck": ",".join(str(card) for card in deck),
        }
    )

deck_summary = pd.DataFrame(deck_rows).sort_values(["games", "wins", "unique_teams"], ascending=False)
deck_summary.to_csv(OUT_DIR / "exact_deck_hashes.csv", index=False)

view_cols = [
    "deck_hash",
    "archetype",
    "games",
    "wins",
    "win_rate",
    "unique_teams",
    "example_teams",
    "first_seen",
    "last_seen",
    "day_counts",
    "top_cards",
]
display(deck_summary[deck_summary["games"] >= MIN_DECK_GAMES_FOR_VIEW][view_cols].head(40))

## Latest Complete Day: Top 50 Exact Decks

This is the evaluation-weight source. It ranks exact lists by player-deck count on the latest complete day only, while retaining the two-day columns as a stability check.

In [ ]:
LATEST_DATE = analysis_dates[-1]
latest_records = records_df[records_df["date"] == LATEST_DATE].copy()
if latest_records.empty:
    raise RuntimeError(f"No player-deck records found for latest date {LATEST_DATE}")

latest_counts = (
    latest_records.groupby("deck_hash", dropna=False)
    .agg(
        latest_games=("deck_hash", "size"),
        latest_wins=("is_win_num", "sum"),
        latest_unique_teams=("team", "nunique"),
    )
    .reset_index()
)
latest_counts["latest_win_rate"] = latest_counts["latest_wins"] / latest_counts["latest_games"]

deck_meta = deck_summary[
    [
        "deck_hash",
        "archetype",
        "games",
        "wins",
        "win_rate",
        "unique_teams",
        "example_teams",
        "first_seen",
        "last_seen",
        "day_counts",
        "top_cards",
        "deck",
    ]
].rename(
    columns={
        "games": "window_games",
        "wins": "window_wins",
        "win_rate": "window_win_rate",
        "unique_teams": "window_unique_teams",
    }
)

latest_top50 = latest_counts.merge(deck_meta, on="deck_hash", how="left")
latest_top50 = latest_top50.sort_values(
    ["latest_games", "latest_wins", "latest_unique_teams", "deck_hash"],
    ascending=[False, False, False, True],
).head(50).reset_index(drop=True)
latest_top50.insert(0, "rank", range(1, len(latest_top50) + 1))

latest_player_decks = int(len(latest_records))
selected_player_decks = int(latest_top50["latest_games"].sum())
latest_top50["latest_share"] = latest_top50["latest_games"] / latest_player_decks
latest_top50["covered_share"] = latest_top50["latest_games"] / selected_player_decks

latest_archetypes = (
    latest_top50.groupby("archetype", dropna=False)
    .agg(
        exact_decks=("deck_hash", "size"),
        latest_games=("latest_games", "sum"),
        latest_wins=("latest_wins", "sum"),
    )
    .reset_index()
)
latest_archetypes["latest_win_rate"] = (
    latest_archetypes["latest_wins"] / latest_archetypes["latest_games"]
)
latest_archetypes["latest_share"] = latest_archetypes["latest_games"] / latest_player_decks
latest_archetypes["covered_share"] = latest_archetypes["latest_games"] / selected_player_decks
latest_archetypes = latest_archetypes.sort_values("latest_games", ascending=False).reset_index(drop=True)

latest_top50_path = OUT_DIR / "latest_top50_exact_decks.csv"
latest_archetypes_path = OUT_DIR / "latest_top50_archetype_distribution.csv"
latest_top50.to_csv(latest_top50_path, index=False)
latest_archetypes.to_csv(latest_archetypes_path, index=False)

latest_top50_summary = {
    "latest_date": LATEST_DATE,
    "latest_player_decks": latest_player_decks,
    "selected_exact_decks": int(len(latest_top50)),
    "selected_player_decks": selected_player_decks,
    "coverage": selected_player_decks / latest_player_decks,
    "top_archetypes": latest_archetypes.to_dict(orient="records"),
    "top_exact_decks": latest_top50.head(20).to_dict(orient="records"),
}
(OUT_DIR / "latest_top50_summary.json").write_text(
    json.dumps(latest_top50_summary, ensure_ascii=False, indent=2), encoding="utf-8"
)

print("latest_date:", LATEST_DATE)
print("latest_player_decks:", latest_player_decks)
print("top50_coverage:", round(selected_player_decks / latest_player_decks, 6))
print("latest_top50_path:", latest_top50_path)
display(latest_archetypes)
display(
    latest_top50[
        [
            "rank",
            "deck_hash",
            "archetype",
            "latest_games",
            "latest_share",
            "latest_win_rate",
            "latest_unique_teams",
            "window_games",
            "window_win_rate",
            "example_teams",
        ]
    ].head(50)
)

## Export Top Deck Lists

The exported CSVs are small 60-card-count lists for direct local import and pressure-pool construction.

In [ ]:
deck_out_dir = OUT_DIR / "top_deck_lists"
deck_out_dir.mkdir(parents=True, exist_ok=True)
manifest_rows: list[dict[str, Any]] = []

for rank, row in enumerate(deck_summary.head(TOP_N_DECK_EXPORT).itertuples(index=False), start=1):
    deck = deck_lookup[str(row.deck_hash)]
    counts = Counter(deck)
    safe_name = safe_slug(str(row.archetype))
    deck_path = deck_out_dir / f"rank{rank:02d}_{row.deck_hash}_{safe_name}.csv"
    deck_list = pd.DataFrame(
        [
            {
                "card_id": card_id,
                "card_name": card_names.get(card_id, str(card_id)),
                "count": count,
            }
            for card_id, count in sorted(counts.items(), key=lambda item: (-item[1], item[0]))
        ]
    )
    deck_list.to_csv(deck_path, index=False)
    manifest_rows.append(
        {
            "rank": rank,
            "deck_hash": row.deck_hash,
            "archetype": row.archetype,
            "games": row.games,
            "wins": row.wins,
            "win_rate": row.win_rate,
            "unique_teams": row.unique_teams,
            "example_teams": row.example_teams,
            "path": str(deck_path),
        }
    )

deck_manifest = pd.DataFrame(manifest_rows)
deck_manifest.to_csv(deck_out_dir / "top_deck_manifest.csv", index=False)
display(deck_manifest.head(20))
print("exported:", len(deck_manifest), "deck lists to", deck_out_dir)

## Team Audit

This table approximates the visible top-ladder field in the official episode dumps. Teams with multiple exact decks are especially useful for detecting fast adaptation.

In [ ]:
team_rows: list[dict[str, Any]] = []
for team, group in records_df[records_df["team"].astype(str) != ""].groupby("team"):
    archetype_counts = Counter(group["archetype"])
    deck_counts = Counter(group["deck_hash"])
    day_counts = Counter(group["date"])
    games = len(group)
    wins = int(group["is_win_num"].sum())
    team_rows.append(
        {
            "team": team,
            "games": games,
            "wins": wins,
            "win_rate": wins / games if games else 0.0,
            "unique_decks": len(deck_counts),
            "modal_archetype": archetype_counts.most_common(1)[0][0],
            "archetype_mix": "; ".join(f"{k}:{v}" for k, v in archetype_counts.most_common(5)),
            "modal_deck_hash": deck_counts.most_common(1)[0][0],
            "deck_mix": "; ".join(f"{k}:{v}" for k, v in deck_counts.most_common(5)),
            "active_dates": ",".join(sorted(day_counts)),
        }
    )

team_summary = pd.DataFrame(team_rows).sort_values(["games", "wins"], ascending=False)
team_summary.to_csv(OUT_DIR / "team_audit.csv", index=False)

display(team_summary.head(40))
display(team_summary[team_summary["unique_decks"] > 1].head(40))

## Control / LO Slice

Track this slice separately because it affects whether our agents can survive trap, deck-out, hand disruption, and non-prize-race game plans.

In [ ]:
control_pattern = "Great Tusk|Crustle|Cubchoo|Trevenant|Control|Hammer"
control_decks = deck_summary[deck_summary["archetype"].astype(str).str.contains(control_pattern, case=False, regex=True, na=False)].copy()
control_archetypes = archetype_combined[archetype_combined["archetype"].astype(str).str.contains(control_pattern, case=False, regex=True, na=False)].copy()
control_day = archetype_day[archetype_day["archetype"].astype(str).str.contains(control_pattern, case=False, regex=True, na=False)].copy()

control_decks.to_csv(OUT_DIR / "control_exact_decks.csv", index=False)
control_archetypes.to_csv(OUT_DIR / "control_archetypes.csv", index=False)
control_day.to_csv(OUT_DIR / "control_archetypes_by_day.csv", index=False)

display(control_archetypes)
display(control_day.sort_values(["date", "games"], ascending=[False, False]))
display(control_decks[view_cols].head(40))
print("control_player_decks:", int(control_archetypes["games"].sum()) if not control_archetypes.empty else 0)
print("control_share:", float(control_archetypes["games"].sum() / len(records_df)) if not control_archetypes.empty else 0.0)

## Tech Card Adoption

This view is intentionally card-level, not archetype-level. It catches public adoption of counter cards before the broad classifier changes.

In [ ]:
card_rows: list[dict[str, Any]] = []
for date, group in records_df.groupby("date"):
    total = len(group)
    card_counter: Counter[int] = Counter()
    win_counter: Counter[int] = Counter()
    for row in group.itertuples():
        cards = set(row.deck)
        card_counter.update(cards)
        if row.is_win is True:
            win_counter.update(cards)
    for card_id, games in card_counter.items():
        card_rows.append(
            {
                "date": date,
                "card_id": card_id,
                "card_name": card_names.get(card_id, str(card_id)),
                "games": games,
                "share": games / total if total else 0.0,
                "winner_games": win_counter.get(card_id, 0),
                "winner_share": win_counter.get(card_id, 0) / games if games else 0.0,
            }
        )

card_day = pd.DataFrame(card_rows)
card_day.to_csv(OUT_DIR / "card_presence_by_day.csv", index=False)

interesting_keywords = [
    "Great Tusk",
    "Dwebble",
    "Crustle",
    "Neutralization Zone",
    "Xerosic",
    "Boss",
    "Crushing Hammer",
    "Enhanced Hammer",
    "Buddy-Buddy Poffin",
    "Marnie",
    "Grimmsnarl",
    "Starmie",
    "Lucario",
    "Alakazam",
    "Trevenant",
    "Archaludon",
]
pattern = "|".join(re.escape(k) for k in interesting_keywords)
interesting_cards = card_day[card_day["card_name"].astype(str).str.contains(pattern, case=False, regex=True, na=False)]
interesting_cards = interesting_cards.sort_values(["date", "share"], ascending=[False, False])
interesting_cards.to_csv(OUT_DIR / "interesting_card_presence_by_day.csv", index=False)

display(interesting_cards.head(100))

## Effort-Aware Deck Evidence

Observed win rate is a product of the deck list, policy quality, matchup sampling, and development effort. The tables below cap each team's contribution at 100 games, measure concentration with effective team count, remove each team in turn, and estimate cross-team heterogeneity. They are diagnostics, not causal deck-strength estimates.

In [ ]:
import math


def wilson_lower(wins: int, games: int, z: float = 1.959963984540054) -> float:
    if games <= 0:
        return float("nan")
    p = wins / games
    z2 = z * z
    center = p + z2 / (2 * games)
    radius = z * math.sqrt((p * (1 - p) + z2 / (4 * games)) / games)
    return (center - radius) / (1 + z2 / games)


def dl_tau(team_table: pd.DataFrame, min_team_games: int = 30) -> float:
    eligible = team_table[team_table["games"] >= min_team_games].copy()
    if len(eligible) < 2:
        return float("nan")
    rates = eligible["wins"] / eligible["games"]
    variances = (rates * (1 - rates) / eligible["games"]).clip(lower=1e-8)
    weights = 1 / variances
    fixed_mean = float((weights * rates).sum() / weights.sum())
    q = float((weights * (rates - fixed_mean) ** 2).sum())
    c = float(weights.sum() - (weights**2).sum() / weights.sum())
    tau2 = max(0.0, (q - (len(eligible) - 1)) / c) if c > 0 else 0.0
    return math.sqrt(tau2)


def effort_summary(frame: pd.DataFrame, key_cols: list[str]) -> pd.DataFrame:
    work = frame.copy()
    work["team_norm"] = work["team"].fillna("").astype(str).str.strip().str.casefold()
    work.loc[work["team_norm"] == "", "team_norm"] = "<unknown>"
    team = (
        work.groupby(key_cols + ["team_norm"], dropna=False)
        .agg(games=("deck_hash", "size"), wins=("is_win_num", "sum"))
        .reset_index()
    )
    daily = (
        work.groupby(key_cols + ["date"], dropna=False)
        .agg(games=("deck_hash", "size"), wins=("is_win_num", "sum"))
        .reset_index()
    )
    daily["win_rate"] = daily["wins"] / daily["games"]

    rows: list[dict[str, Any]] = []
    grouper: Any = key_cols[0] if len(key_cols) == 1 else key_cols
    for raw_keys, group in team.groupby(grouper, dropna=False):
        keys = raw_keys if isinstance(raw_keys, tuple) else (raw_keys,)
        row = dict(zip(key_cols, keys))
        games = int(group["games"].sum())
        wins = int(group["wins"].sum())
        shares = group["games"] / games
        capped_games = group["games"].clip(upper=100)
        capped_wins = capped_games * group["wins"] / group["games"]
        leave_one_out = [
            (wins - int(item.wins)) / (games - int(item.games))
            for item in group.itertuples(index=False)
            if games > int(item.games)
        ]
        mask = pd.Series(True, index=daily.index)
        for column, value in row.items():
            mask &= daily[column].eq(value)
        daily_slice = daily[mask]
        values = [
            wilson_lower(wins, games),
            float(daily_slice["win_rate"].min()),
            float(capped_wins.sum() / capped_games.sum()),
        ]
        if leave_one_out:
            values.append(min(leave_one_out))
        row.update(
            {
                "games": games,
                "wins": wins,
                "win_rate": wins / games,
                "wilson_lcb95": wilson_lower(wins, games),
                "teams": int(len(group)),
                "effective_teams": float(1 / (shares**2).sum()),
                "largest_team_share": float(shares.max()),
                "cap100_win_rate": float(capped_wins.sum() / capped_games.sum()),
                "worst_leave_one_team_out": min(leave_one_out) if leave_one_out else float("nan"),
                "min_daily_win_rate": float(daily_slice["win_rate"].min()),
                "team_tau": dl_tau(group),
                "pressure_floor": min(values),
            }
        )
        rows.append(row)
    return pd.DataFrame(rows)


known_records = records_df[records_df["known_result"] == True].copy()  # noqa: E712
effort_archetypes = effort_summary(known_records, ["archetype"])
effort_exact = effort_summary(known_records, ["deck_hash", "archetype"])

effort_archetypes = effort_archetypes.sort_values(
    ["pressure_floor", "games"], ascending=[False, False]
)
effort_exact = effort_exact.sort_values(["games", "win_rate"], ascending=[False, False])

effort_archetypes.to_csv(OUT_DIR / "effort_aware_archetypes.csv", index=False)
effort_exact.to_csv(OUT_DIR / "effort_aware_exact_decks.csv", index=False)

candidate_routes = [
    "Crustle",
    "Mega Starmie ex",
    "Cynthia Garchomp",
    "Dudunsparce/Alakazam",
    "Mega Lucario ex",
    "Dragapult ex",
    "Team Rocket Spidops",
]
candidate_hashes = [
    "a3c0bb2839c6",
    "6f87e9d4949a",
    "a4f0650bcc01",
    "9870485d0346",
    "c7b3253feaa4",
    "b1cb1f7bb6cb",
]

effort_route_view = effort_archetypes[effort_archetypes["archetype"].isin(candidate_routes)]
effort_exact_view = effort_exact[effort_exact["deck_hash"].isin(candidate_hashes)]
display(effort_route_view)
display(effort_exact_view)

## Exact-Deck Matchup Shape

Aggregate score can hide a polarized deck. This joins both player records from each episode and reports the opponent archetype for each candidate exact hash. The `games >= 40` floor is descriptive: team policy and opponent quality remain confounded.

In [ ]:
opponent_columns = records_df[
    ["episode_id", "player_index", "team", "deck_hash", "archetype"]
].rename(
    columns={
        "player_index": "opponent_player_index",
        "team": "opponent_team",
        "deck_hash": "opponent_deck_hash",
        "archetype": "opponent_archetype",
    }
)

paired_records = records_df.merge(opponent_columns, on="episode_id", how="inner")
paired_records = paired_records[
    paired_records["player_index"] != paired_records["opponent_player_index"]
].copy()
paired_candidates = paired_records[paired_records["deck_hash"].isin(candidate_hashes)].copy()

candidate_exact_matchups = (
    paired_candidates.groupby(
        ["deck_hash", "archetype", "opponent_archetype"], dropna=False
    )
    .agg(games=("episode_id", "size"), wins=("is_win_num", "sum"))
    .reset_index()
)
candidate_exact_matchups["win_rate"] = (
    candidate_exact_matchups["wins"] / candidate_exact_matchups["games"]
)
candidate_exact_matchups["wilson_lcb95"] = candidate_exact_matchups.apply(
    lambda row: wilson_lower(int(row.wins), int(row.games)), axis=1
)
candidate_exact_matchups = candidate_exact_matchups.sort_values(
    ["deck_hash", "games"], ascending=[True, False]
)
candidate_exact_matchups.to_csv(OUT_DIR / "candidate_exact_matchups.csv", index=False)

major_matchups = candidate_exact_matchups[candidate_exact_matchups["games"] >= 40]
candidate_major_floor = (
    major_matchups.groupby(["deck_hash", "archetype"], dropna=False)
    .agg(
        covered_matchups=("opponent_archetype", "nunique"),
        observed_major_floor=("win_rate", "min"),
        lcb_major_floor=("wilson_lcb95", "min"),
    )
    .reset_index()
    .sort_values("observed_major_floor", ascending=False)
)
candidate_major_floor.to_csv(OUT_DIR / "candidate_major_matchup_floor.csv", index=False)

display(candidate_major_floor)
display(candidate_exact_matchups[candidate_exact_matchups["games"] >= 15])

## Card-Text Structural Audit

This prior is deliberately separate from leaderboard performance. It records what each exact list can do according to its card text: setup path, Prize map, Energy durability, recovery, wall interaction, and the number of decisions the policy must coordinate. Categorical coverage is not converted into a synthetic win-rate score.

In [ ]:
deck_structure_audit = pd.DataFrame(
    [
        {
            "route": "Mega Lucario ex",
            "deck_hash": "b1cb1f7bb6cb",
            "basic_count": 10,
            "setup_and_search": "Stage 1, 4 Riolu / 4 Mega Lucario; Fighting Gong; Dusk Ball bottom-7 search",
            "prize_plan": "Mega ex is the real attacker (3 Prizes); Solrock is only a low-output single-Prize fallback",
            "attack_and_wall_interaction": "130 for 1 Energy or 270 for 2 with a next-turn lock; ex walls can blank the main route",
            "energy_resilience": "10 Basic Fighting + 3 Rock Fighting; good into Enhanced Hammer",
            "resource_loop": "Lunatone discards Basic Energy to draw 3; Aura Jab restores up to 3 Basic Energy to the Bench; Wally heals and returns Energy",
            "bench_resilience": "3 Solrock + 3 Lunatone are exposed 110 HP engine pieces",
            "policy_complexity": "Medium: attack cooldown, Aura Jab allocation, Wally reset, turn order, and matchup modes",
            "intrinsic_risks": "Three-Prize race, ex wall / Neutralization Zone, bench-engine pressure, and mill pacing",
        },
        {
            "route": "Crustle",
            "deck_hash": "6f87e9d4949a",
            "basic_count": 9,
            "setup_and_search": "Stage 1, 4-4 line; 70 HP Dwebble is Poffin-searchable and Ascension evolves for 1 Colorless",
            "prize_plan": "Single-Prize Crustle wall plus 3-Prize Mega Kangaskhan pivot",
            "attack_and_wall_interaction": "Crustle blocks ex damage and Superb Scissors ignores effects; fixed 120 damage is slow into large bodies",
            "energy_resilience": "1 Basic Grass + 12 Special Energy; both real attacks cost 3 and there is no acceleration or recovery",
            "resource_loop": "Active Kangaskhan draws 2; Lillie recycles the hand; no discard recovery",
            "bench_resilience": "Shaymin blocks attack damage and 2 Battle Cage block opponent damage counters to the Bench",
            "policy_complexity": "Low-medium: choose Crustle wall versus Kangaskhan race, then order Energy, Switch, heal, and disruption",
            "intrinsic_risks": "Damage counters / effect-piercing attacks, Special Energy denial, and a polarized 120-damage race",
        },
        {
            "route": "Dudunsparce/Alakazam",
            "deck_hash": "a4f0650bcc01",
            "basic_count": 9,
            "setup_and_search": "Stage 2, 4-4-4 line; 3 Rare Candy, Poffin, Dawn, Hilda, and Telepath Energy provide redundancy",
            "prize_plan": "Alakazam is a single-Prize main attacker; Fezandipiti ex is an optional liability",
            "attack_and_wall_interaction": "Places 2 counters per hand card and bypasses damage walls, but Mist Energy can prevent this attack effect",
            "energy_resilience": "2 Basic Psychic + 4 Telepath + 1 Enriching; only 1 Energy to attack, but key setup Energy is special",
            "resource_loop": "Dudunsparce draw-and-shuffle, evolution draw, Sacred Ash, Lana, and Night Stretcher",
            "bench_resilience": "Shaymin blocks attack damage to non-Rule-Box Bench; Abra remains a fragile 50 HP setup point",
            "policy_complexity": "High: hand size is damage, so draw, evolution, disruption, recycling, and discard choices are coupled",
            "intrinsic_risks": "Hand compression, early Abra removal, and self-decking when the draw engine is overused",
        },
        {
            "route": "Mega Starmie ex",
            "deck_hash": "9870485d0346",
            "basic_count": 6,
            "setup_and_search": "Stage 1, 4 Staryu / 3 Starmie with Poffin and Ultra Ball; 2-2-2 Dusknoir line has no Rare Candy",
            "prize_plan": "3-Prize Mega attacker; every Cursed Blast also concedes a Prize",
            "attack_and_wall_interaction": "120 plus 50 Bench damage for 1 Water; 210 Nebula Beam ignores effects on the Active Pokemon",
            "energy_resilience": "9 Basic Water + 4 Ignition; repeated attack is basic-energy anchored and burst Energy is special",
            "resource_loop": "Lillie and Judge reset hands; Wally fully heals but returns attached Energy; no Pokemon recovery",
            "bench_resilience": "No Shaymin; Staryu and the Dusknoir line are exposed setup targets",
            "policy_complexity": "Medium-high: Bench target, self-KO Prize timing, Wally, and same-turn Ignition must agree",
            "intrinsic_risks": "Three-Prize body plus self-Prizes, only 6 Basics, 0 Switch / 0 Boss, and limited rebuild after Staryu losses",
        },
        {
            "route": "Cynthia Garchomp",
            "deck_hash": "c7b3253feaa4",
            "basic_count": 10,
            "setup_and_search": "Stage 2, 4-4-3 without Rare Candy; Gabite searches Cynthia Pokemon; Poffin, Gong, and Roserade support",
            "prize_plan": "2-Prize Garchomp ex plus conditional single-Prize Spiritomb route",
            "attack_and_wall_interaction": "100 and draw to 6 or 260 and discard Energy; Roserade adds 30 each; no unconditional wall-piercing hit",
            "energy_resilience": "5 Basic Fighting + 4 Rock Fighting; 1-Energy fallback and Fighting Gong are Hammer-resilient",
            "resource_loop": "Gabite searches every turn, Garchomp refills by attacking, plus 2 Night Stretcher and Lillie",
            "bench_resilience": "Gabite and Roserade are required Bench engines; Power Weight adds 70 HP to Cynthia Pokemon",
            "policy_complexity": "High: two evolution lines, Roserade thresholds, Spiritomb, and the all-Energy discard attack interact",
            "intrinsic_risks": "Setup / Bench disruption, recharge after Draconic Buster, and no unconditional wall-piercing attack",
        },
    ]
)

deck_structure_audit["opening_basic_probability"] = deck_structure_audit["basic_count"].map(
    lambda basics: 1 - math.comb(60 - basics, 7) / math.comb(60, 7)
)
deck_structure_audit["expected_mulligans"] = (
    1 - deck_structure_audit["opening_basic_probability"]
) / deck_structure_audit["opening_basic_probability"]

counter_coverage = pd.DataFrame(
    [
        ["Mega Lucario ex", "No clean main-deck answer", "Strong", "Medium", "Fragile", "Fragile", "Poor"],
        ["Crustle", "Strong into ex; weak to counters", "Fragile", "Medium", "Medium", "Hybrid", "Polarized"],
        ["Dudunsparce/Alakazam", "Strong unless Mist Energy remains", "Medium", "Fragile", "Medium", "Strong", "Scales with hand"],
        ["Mega Starmie ex", "Strong via Nebula; regression-test Zone", "Strong", "Medium", "Fragile", "Poor", "210 + Cursed Blast"],
        ["Cynthia Garchomp", "Conditional Spiritomb only", "Strong", "Strong after attack", "Medium", "Strong", "260 + Roserade"],
    ],
    columns=[
        "route",
        "ex_wall_or_zone",
        "special_energy_denial",
        "hand_compression",
        "mill_stall",
        "prize_exchange",
        "high_hp_threshold",
    ],
)

deck_structure_audit.to_csv(OUT_DIR / "deck_structure_audit.csv", index=False)
counter_coverage.to_csv(OUT_DIR / "deck_counter_coverage.csv", index=False)
display(deck_structure_audit)
display(counter_coverage)

## Portfolio Decision

- Keep the mature Lucario agent as the production champion; its score does not prove that Lucario is the best underlying deck.
- Use Starmie `987` as the primary finite-budget, low-hard-lock challenger. Its first tests must address the six-Basic opening risk and the zero-Switch / zero-Boss control blind spot.
- Use Crustle `6f87` as the fastest transferable benchmark, not as an automatic replacement: its cross-team evidence is strongest, but its Special-Energy and Starmie matchups are sharply polarized.
- Keep Alakazam `a4f` as the long-horizon ceiling route. Its single-Prize counter-placement plan is excellent, but Mist Energy, hand compression, and policy complexity create a low implementation floor.
- Keep Cynthia `c7` as the secondary balanced route. Its two-Prize map and durable Energy are attractive, but the public result is too concentrated in one team to call independently reproduced.

## Equal-Budget Challenger Protocol

Phase A gives Crustle, Alakazam, Starmie, and Cynthia the same small budget: the same stratified replay audit, four policy iterations, candidate count, paired `rep360` seeds, and loss-review count. Lucario is reported as the mature champion, not as an equal-effort control.

Phase B advances the two best learning curves. Each receives eight total policy iterations, the same Transformer state-action tokens, PPO environment steps, model size, hyperparameter trials, and three training seeds. The locked test is 12 opponents x 2 seats x 50 shared seeds (1,200 games), plus 6 weakness opponents x 2 seats x 40 seeds (480 games).

Report weighted win rate, Wilson lower bound, worst-quartile matchup mean, turn-order gap, no-attack rate, empty-board rate, deck-out rate, and variance across training seeds. Promotion requires both a competitive mean and a better failure floor; a high aggregate from one favorable matchup is insufficient.

## Notebook Summary

Download `notebook_summary.json` plus `top_deck_lists/top_deck_manifest.csv` after the Kaggle run. Those two files are enough to decide which exact decks to import for local pressure testing.

In [ ]:
summary = {
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "target_dates": TARGET_DATES,
    "analysis_dates": analysis_dates,
    "missing_dates": missing_dates,
    "json_parse_workers": JSON_PARSE_WORKERS,
    "day_file_counts": day_file_counts,
    "player_deck_records": int(len(records_df)),
    "unique_episodes": int(records_df["episode_id"].nunique()),
    "unique_decks": int(records_df["deck_hash"].nunique()),
    "latest_top50": latest_top50_summary,
    "top_archetypes": archetype_combined.head(15).to_dict(orient="records"),
    "archetype_trend": trend.head(25).to_dict(orient="records"),
    "archetype_robustness": robust_candidate_view.head(25).to_dict(orient="records"),
    "top_exact_decks": deck_summary[view_cols].head(20).to_dict(orient="records"),
    "effort_aware_archetypes": effort_route_view.to_dict(orient="records"),
    "effort_aware_exact_decks": effort_exact_view.to_dict(orient="records"),
    "candidate_major_matchup_floor": candidate_major_floor.to_dict(orient="records"),
    "top_teams": team_summary.head(20).to_dict(orient="records"),
    "control_slice": {
        "games": int(control_archetypes["games"].sum()) if not control_archetypes.empty else 0,
        "share": float(control_archetypes["games"].sum() / len(records_df)) if not control_archetypes.empty else 0.0,
        "archetypes": control_archetypes.to_dict(orient="records"),
    },
    "outputs": {
        "out_dir": str(OUT_DIR),
        "records": str(records_path),
        "archetype_combined": str(OUT_DIR / "archetype_combined.csv"),
        "archetype_by_day": str(OUT_DIR / "archetype_by_day.csv"),
        "archetype_trend": str(OUT_DIR / "archetype_trend.csv"),
        "archetype_robustness": str(OUT_DIR / "archetype_robustness.csv"),
        "exact_deck_hashes": str(OUT_DIR / "exact_deck_hashes.csv"),
        "latest_top50_exact_decks": str(latest_top50_path),
        "latest_top50_archetypes": str(latest_archetypes_path),
        "effort_aware_archetypes": str(OUT_DIR / "effort_aware_archetypes.csv"),
        "effort_aware_exact_decks": str(OUT_DIR / "effort_aware_exact_decks.csv"),
        "candidate_exact_matchups": str(OUT_DIR / "candidate_exact_matchups.csv"),
        "candidate_major_matchup_floor": str(OUT_DIR / "candidate_major_matchup_floor.csv"),
        "deck_structure_audit": str(OUT_DIR / "deck_structure_audit.csv"),
        "deck_counter_coverage": str(OUT_DIR / "deck_counter_coverage.csv"),
        "team_audit": str(OUT_DIR / "team_audit.csv"),
        "top_deck_manifest": str(deck_out_dir / "top_deck_manifest.csv"),
        "control_exact_decks": str(OUT_DIR / "control_exact_decks.csv"),
        "interesting_card_presence_by_day": str(OUT_DIR / "interesting_card_presence_by_day.csv"),
    },
}

summary_path = OUT_DIR / "notebook_summary.json"
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print("summary_path:", summary_path)
print(json.dumps({k: summary[k] for k in ["target_dates", "player_deck_records", "unique_episodes", "unique_decks"]}, indent=2))
display(pd.DataFrame(summary["top_archetypes"]).head(15))
display(pd.DataFrame(summary["archetype_robustness"]).head(15))
display(pd.DataFrame(summary["top_exact_decks"]).head(15))

## Decision Use

- If Great Tusk / Crustle / Hammer control share is rising, increase its weight in local gates before promoting PPO checkpoints.
- If a new exact deck hash has both high volume and high win rate, import that exported list into the pressure pool before changing our learner deck.
- If a top team rotates multiple exact decks, treat the modal deck as less stable and validate against the whole archetype family.

## Visual Strength Dashboard

This section ranks archetypes with enough data to be useful, attaches a representative official card image to each major archetype, and exports the tables and figures for Kaggle download.

In [ ]:
import html
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, display
from matplotlib.ticker import PercentFormatter

REPRESENTATIVE_CARDS = {
    "Marnie's Grimmsnarl ex": {
        'card_id': 648,
        'pokemon': "Marnie's Grimmsnarl ex",
        'image_url': 'https://images.pokemontcg.io/sv10/136.png',
    },
    'Dudunsparce/Alakazam': {
        'card_id': 743,
        'pokemon': 'Alakazam',
        'image_url': 'https://images.pokemontcg.io/me1/56.png',
    },
    'Team Rocket Spidops': {
        'card_id': 401,
        'pokemon': "Team Rocket's Spidops",
        'image_url': 'https://images.pokemontcg.io/sv10/20.png',
    },
    'Crustle': {
        'card_id': 345,
        'pokemon': 'Crustle',
        'image_url': 'https://images.pokemontcg.io/sv10/12.png',
    },
    'Cynthia Garchomp': {
        'card_id': 381,
        'pokemon': "Cynthia's Garchomp ex",
        'image_url': 'https://images.pokemontcg.io/sv10/104.png',
    },
    'Dragapult ex': {
        'card_id': 121,
        'pokemon': 'Dragapult ex',
        'image_url': 'https://images.pokemontcg.io/sv6/130.png',
    },
}

MAIN_ARCHETYPES = [
    "Marnie's Grimmsnarl ex",
    'Dudunsparce/Alakazam',
    'Team Rocket Spidops',
    'Crustle',
    'Cynthia Garchomp',
    'Dragapult ex',
]
MIN_GAMES_FOR_STRENGTH = 100

strength_ranking = archetype_combined.copy()
strength_ranking['wilson_lcb95'] = strength_ranking.apply(
    lambda row: wilson_lower(int(row.wins), int(row.games)), axis=1
)
pressure_floor_map = effort_archetypes.set_index('archetype')['pressure_floor']
strength_ranking['pressure_floor'] = strength_ranking['archetype'].map(pressure_floor_map)
strength_ranking = strength_ranking[
    (strength_ranking['games'] >= MIN_GAMES_FOR_STRENGTH)
    & (strength_ranking['archetype'] != 'Unknown')
].sort_values(
    ['wilson_lcb95', 'games', 'win_rate'], ascending=[False, False, False]
).reset_index(drop=True)
strength_ranking.insert(0, 'strength_rank', range(1, len(strength_ranking) + 1))
strength_ranking.to_csv(OUT_DIR / 'archetype_strength_ranking.csv', index=False)

strength_view = strength_ranking[
    ['strength_rank', 'archetype', 'games', 'wins', 'win_rate', 'wilson_lcb95', 'share', 'unique_teams', 'pressure_floor']
].copy()
display(strength_view.style.format({
    'win_rate': '{:.1%}',
    'wilson_lcb95': '{:.1%}',
    'share': '{:.1%}',
    'pressure_floor': '{:.1%}',
}))

plot_frame = strength_ranking.sort_values('wilson_lcb95', ascending=True)
fig, ax = plt.subplots(figsize=(10.5, 5.6))
y_positions = np.arange(len(plot_frame))
ax.barh(y_positions, plot_frame['win_rate'], color='#4f86c6', alpha=0.88, label='Observed win rate')
ax.scatter(plot_frame['wilson_lcb95'], y_positions, color='#c43d3d', s=58, zorder=3, label='95% Wilson lower bound')
for y_position, row in zip(y_positions, plot_frame.itertuples()):
    ax.text(
        min(float(row.win_rate) + 0.006, 0.67),
        y_position,
        f'{row.win_rate:.1%}  (n={int(row.games)})',
        va='center',
        fontsize=9,
    )
ax.axvline(0.5, color='#777777', linestyle='--', linewidth=1, label='50% reference')
ax.set_yticks(y_positions)
ax.set_yticklabels(plot_frame['archetype'])
ax.set_xlim(0, max(0.62, float(plot_frame['win_rate'].max()) + 0.09))
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xlabel('Win rate')
ax.set_title('Two-day archetype strength: volume-qualified ranking')
ax.legend(loc='lower right', frameon=False)
ax.grid(axis='x', alpha=0.2)
fig.tight_layout()
fig.savefig(OUT_DIR / 'archetype_strength_ranking.png', dpi=160, bbox_inches='tight')
plt.show()
plt.close(fig)

exact_strength_ranking = deck_summary.merge(
    effort_exact[['deck_hash', 'cap100_win_rate', 'pressure_floor']],
    on='deck_hash',
    how='left',
)
exact_strength_ranking['wilson_lcb95'] = exact_strength_ranking.apply(
    lambda row: wilson_lower(int(row.wins), int(row.games)), axis=1
)
exact_strength_ranking = exact_strength_ranking[
    (exact_strength_ranking['games'] >= MIN_GAMES_FOR_STRENGTH)
    & (exact_strength_ranking['archetype'] != 'Unknown')
].sort_values(
    ['wilson_lcb95', 'games', 'win_rate'], ascending=[False, False, False]
).reset_index(drop=True)
exact_strength_ranking.insert(0, 'strength_rank', range(1, len(exact_strength_ranking) + 1))
exact_strength_ranking.to_csv(OUT_DIR / 'exact_deck_strength_ranking.csv', index=False)
display(exact_strength_ranking[
    ['strength_rank', 'deck_hash', 'archetype', 'games', 'wins', 'win_rate', 'wilson_lcb95', 'unique_teams', 'cap100_win_rate']
].head(15).style.format({
    'win_rate': '{:.1%}',
    'wilson_lcb95': '{:.1%}',
    'cap100_win_rate': '{:.1%}',
}))

representative_rows = []
for row in strength_ranking[strength_ranking['archetype'].isin(MAIN_ARCHETYPES)].itertuples():
    meta = REPRESENTATIVE_CARDS.get(row.archetype)
    if meta is None:
        continue
    representative_rows.append({
        'rank': int(row.strength_rank),
        'archetype': row.archetype,
        'representative_card_id': meta['card_id'],
        'representative_pokemon': meta['pokemon'],
        'image_url': meta['image_url'],
        'games': int(row.games),
        'win_rate': float(row.win_rate),
        'wilson_lcb95': float(row.wilson_lcb95),
    })
representative_cards = pd.DataFrame(representative_rows)
representative_cards.to_csv(OUT_DIR / 'representative_deck_cards.csv', index=False)

html_rows = []
for row in representative_cards.itertuples():
    meta = REPRESENTATIVE_CARDS[row.archetype]
    image_url = html.escape(meta['image_url'], quote=True)
    alt_text = html.escape(meta['pokemon'], quote=True)
    image_html = "<img src='{}' alt='{}' style='height:150px; max-width:190px; object-fit:contain;'>".format(image_url, alt_text)
    html_rows.append(
        "<tr><td>{}</td><td><b>{}</b></td><td>{}</td><td>{}</td><td>{:,}</td><td>{:.1%}</td><td>{:.1%}</td></tr>".format(
            row.rank, html.escape(row.archetype), image_html, html.escape(row.representative_pokemon),
            row.games, row.win_rate, row.wilson_lcb95,
        )
    )
display(HTML(
    "<table style='border-collapse:collapse; width:100%; text-align:center'>"
    "<thead><tr><th>Rank</th><th>Archetype</th><th>Representative card</th><th>Pokemon</th>"
    "<th>Games</th><th>Win rate</th><th>Wilson LCB95</th></tr></thead>"
    "<tbody>" + ''.join(html_rows) + "</tbody></table>"
))

impact_frame = archetype_combined[
    (archetype_combined['games'] >= MIN_GAMES_FOR_STRENGTH)
    & (archetype_combined['archetype'] != 'Unknown')
].copy()
fig, ax = plt.subplots(figsize=(10.5, 6.0))
ax.scatter(impact_frame['share'], impact_frame['win_rate'], s=impact_frame['games'] / 3, alpha=0.75, color='#299b85', edgecolor='white', linewidth=0.8)
for row in impact_frame.itertuples():
    ax.annotate(row.archetype, (row.share, row.win_rate), xytext=(5, 5), textcoords='offset points', fontsize=9)
ax.axhline(0.5, color='#777777', linestyle='--', linewidth=1)
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xlabel('Meta share')
ax.set_ylabel('Observed win rate')
ax.set_title('Meta impact versus performance (bubble size = games)')
ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(OUT_DIR / 'meta_share_vs_win_rate.png', dpi=160, bbox_inches='tight')
plt.show()
plt.close(fig)

## Archetype Matchup Matrix

Each cell estimates P(row archetype wins | row archetype vs column archetype) from the observed two-day player records. The matrix is player-perspective, so the mirror diagonal should be close to 50%. Cells with fewer than 30 games are marked with an asterisk and should not drive a matchup conclusion.

In [ ]:
MAIN_ARCHETYPES = [
    "Marnie's Grimmsnarl ex",
    'Dudunsparce/Alakazam',
    'Team Rocket Spidops',
    'Crustle',
    'Cynthia Garchomp',
    'Dragapult ex',
]
MIN_MATCHUP_GAMES = 30

matrix_source = paired_records[
    paired_records['known_result']
    & paired_records['archetype'].isin(MAIN_ARCHETYPES)
    & paired_records['opponent_archetype'].isin(MAIN_ARCHETYPES)
].copy()

archetype_matchups = (
    matrix_source.groupby(['archetype', 'opponent_archetype'], dropna=False)
    .agg(games=('episode_id', 'size'), wins=('is_win_num', 'sum'))
    .reset_index()
)
archetype_matchups['win_rate'] = archetype_matchups['wins'] / archetype_matchups['games']
archetype_matchups['wilson_lcb95'] = archetype_matchups.apply(
    lambda row: wilson_lower(int(row.wins), int(row.games)), axis=1
)
archetype_matchups.to_csv(OUT_DIR / 'archetype_matchups.csv', index=False)

rate_matrix = archetype_matchups.pivot(
    index='archetype', columns='opponent_archetype', values='win_rate'
).reindex(index=MAIN_ARCHETYPES, columns=MAIN_ARCHETYPES)
count_matrix = archetype_matchups.pivot(
    index='archetype', columns='opponent_archetype', values='games'
).reindex(index=MAIN_ARCHETYPES, columns=MAIN_ARCHETYPES).fillna(0).astype(int)
rate_matrix.to_csv(OUT_DIR / 'archetype_matchup_win_rate_matrix.csv')
count_matrix.to_csv(OUT_DIR / 'archetype_matchup_game_count_matrix.csv')

display_matrix = rate_matrix.copy().astype(object)
for row_label in MAIN_ARCHETYPES:
    for column_label in MAIN_ARCHETYPES:
        rate = rate_matrix.loc[row_label, column_label]
        games = count_matrix.loc[row_label, column_label]
        if pd.isna(rate):
            display_matrix.loc[row_label, column_label] = 'n/a'
        elif games < MIN_MATCHUP_GAMES:
            display_matrix.loc[row_label, column_label] = f'{rate:.1%}* (n={games})'
        else:
            display_matrix.loc[row_label, column_label] = f'{rate:.1%} (n={games})'
display(display_matrix)

heatmap_rates = rate_matrix.where(count_matrix >= MIN_MATCHUP_GAMES)
fig, ax = plt.subplots(figsize=(11.5, 8.5))
image = ax.imshow(heatmap_rates.to_numpy(dtype=float), cmap='RdYlGn', vmin=0.30, vmax=0.70)
ax.set_xticks(np.arange(len(MAIN_ARCHETYPES)))
ax.set_yticks(np.arange(len(MAIN_ARCHETYPES)))
ax.set_xticklabels(MAIN_ARCHETYPES, rotation=35, ha='right')
ax.set_yticklabels(MAIN_ARCHETYPES)
for row_index, row_label in enumerate(MAIN_ARCHETYPES):
    for column_index, column_label in enumerate(MAIN_ARCHETYPES):
        rate = rate_matrix.loc[row_label, column_label]
        games = count_matrix.loc[row_label, column_label]
        if pd.isna(rate):
            label = 'n/a'
        elif games < MIN_MATCHUP_GAMES:
            label = f'{rate:.1%}*\n(n={games})'
        else:
            label = f'{rate:.1%}\n(n={games})'
        ax.text(column_index, row_index, label, ha='center', va='center', fontsize=8)
ax.set_xlabel('Opponent archetype')
ax.set_ylabel('Row archetype')
ax.set_title('Observed archetype matchup win rate (row wins against column)')
colorbar = fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
colorbar.ax.yaxis.set_major_formatter(PercentFormatter(1.0))
fig.tight_layout()
fig.savefig(OUT_DIR / 'archetype_matchup_heatmap.png', dpi=160, bbox_inches='tight')
plt.show()
plt.close(fig)

print('matchup_records:', len(archetype_matchups))
print('matchup_min_games_for_unstarred_cells:', MIN_MATCHUP_GAMES)

## Visualization Outputs

The PNG files and CSV matrices below are saved alongside the existing analysis artifacts for download from Kaggle Output.

In [ ]:
summary_payload = json.loads(summary_path.read_text(encoding='utf-8') )
summary_payload['target_dates'] = TARGET_DATES
summary_payload['analysis_dates'] = analysis_dates
summary_payload['visualizations'] = {
    'strength_ranking': str(OUT_DIR / 'archetype_strength_ranking.csv'),
    'exact_deck_strength_ranking': str(OUT_DIR / 'exact_deck_strength_ranking.csv'),
    'representative_deck_cards': str(OUT_DIR / 'representative_deck_cards.csv'),
    'archetype_matchups': str(OUT_DIR / 'archetype_matchups.csv'),
    'archetype_matchup_win_rate_matrix': str(OUT_DIR / 'archetype_matchup_win_rate_matrix.csv'),
    'archetype_matchup_game_count_matrix': str(OUT_DIR / 'archetype_matchup_game_count_matrix.csv'),
    'archetype_strength_ranking_png': str(OUT_DIR / 'archetype_strength_ranking.png'),
    'meta_share_vs_win_rate_png': str(OUT_DIR / 'meta_share_vs_win_rate.png'),
    'archetype_matchup_heatmap_png': str(OUT_DIR / 'archetype_matchup_heatmap.png'),
}
summary_path.write_text(json.dumps(summary_payload, ensure_ascii=False, indent=2), encoding='utf-8')
print('visualization_outputs:', len(summary_payload['visualizations']))